# Phase OOS Validation

**Design reference:** `docs/PHASE_OOS_DESIGN.md`  
**Pre-registration:** All thresholds hardcoded from training window before test window is opened.  

---
> **Instructions:** Run Cell 1 first. It will fail with an AssertionError if any learned constant
> has not been filled in. Do not view test results before passing the firewall. Do not adjust
> thresholds after viewing test results.

---

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 1 — DATA FIREWALL
#
# All learned values are derived from train window (≤ 2007Q4)
# only and hardcoded here as constants. This cell must be run
# before any data is loaded. Do not modify constants after
# opening the test window.
# ════════════════════════════════════════════════════════════
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import display

# Ensure project root (containing src/) is on sys.path.
# Works whether the notebook is run via Jupyter Lab or nbconvert.
_nb_root = Path.cwd()
for _candidate in [_nb_root, _nb_root.parent, _nb_root.parent.parent]:
    if (_candidate / "src").is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# ── TRAIN / TEST BOUNDARY ────────────────────────────────────
TRAIN_END  = pd.Timestamp("2007-12-31")
TEST_START = pd.Timestamp("2008-03-31")

# ── FIXED PRIOR (structural economic boundary; not data-derived) ─
# Zero real rate separates financial repression from conventional
# monetary conditions. No estimation required.
REAL_RATE_THRESHOLD = 0.0

# ── LEARNED FROM TRAIN WINDOW ONLY ───────────────────────────
# DTI_FRAGILITY_CUTOFF
#   65th percentile of DTI in regime=0 (adverse) training observations.
#   Derived from: build_master_df() → train split → regime==0 rows → quantile(0.65)
#   See: PHASE_OOS_DESIGN.md §3.2
DTI_FRAGILITY_CUTOFF = 99.163

# PATH_A_LOGIT_COEFS
#   statsmodels Logit coefficients from fit_interaction_logit() on
#   labeled training data (≤ 2007-12-31) only.
#   Features: const, dti, regime, dti_x_regime (= dti * regime)
#   See: PHASE_OOS_DESIGN.md §3.3
PATH_A_LOGIT_COEFS = {
    "const":        -7.094508,
    "dti":           0.071753,
    "regime":       -8.917780,
    "dti_x_regime":  0.096797,
}

# ── ASSERTIONS ───────────────────────────────────────────────
assert REAL_RATE_THRESHOLD == 0.0, \
    "REAL_RATE_THRESHOLD must be 0.0 (fixed structural prior)"

assert DTI_FRAGILITY_CUTOFF is not None and isinstance(DTI_FRAGILITY_CUTOFF, (int, float)), (
    "DTI_FRAGILITY_CUTOFF not set. "
    "Derive from train window: regime==0 training rows, quantile(0.65) of DTI."
)

assert PATH_A_LOGIT_COEFS is not None and isinstance(PATH_A_LOGIT_COEFS, dict), (
    "PATH_A_LOGIT_COEFS not set. "
    "Fit fit_interaction_logit() on train window data only."
)
assert set(PATH_A_LOGIT_COEFS) >= {"const", "dti", "regime", "dti_x_regime"}, (
    f"PATH_A_LOGIT_COEFS missing keys. Got: {set(PATH_A_LOGIT_COEFS)}"
)

# ── PRINT FROZEN CONSTANTS ───────────────────────────────────
print("=" * 62)
print("DATA FIREWALL PASSED — CONSTANTS LOCKED")
print("=" * 62)
print(f"  REAL_RATE_THRESHOLD  = {REAL_RATE_THRESHOLD}  (fixed prior, not learned)")
print(f"  DTI_FRAGILITY_CUTOFF = {DTI_FRAGILITY_CUTOFF}")
print(f"  PATH_A_LOGIT_COEFS:")
for k, v in PATH_A_LOGIT_COEFS.items():
    print(f"    {k:<16s} = {v:+.6f}")
print(f"  Train window ends    : {TRAIN_END.date()}")
print(f"  Test window starts   : {TEST_START.date()}")
print("=" * 62)
print("No test data has been loaded above this line.")

---
### Load data

Builds the full synthetic panel (needed for rolling `dti_pct_roll` context), then filters to the test window.  
The firewall assertion on date bounds is enforced at the end of this cell.

In [ ]:
# ── Cell 2: Load and validate test data ──────────────────────
from src.research.path_a.build_dataset import build_master_df
from src.research.path_a.label_correction import add_correction_label

_ROLL_W = 20  # must match build_dataset._ROLL_W

# Build full panel (1992–2024); rolling context requires full history.
_master = build_master_df()

# Add dti_pct_roll inline — same formula as build_dataset.py.
_dti_arr = _master["dti"].to_numpy()
_n = len(_dti_arr)
_pct = np.empty(_n)
for _t in range(_n):
    _s = max(0, _t - _ROLL_W + 1)
    _seg = _dti_arr[_s : _t + 1]
    _pct[_t] = np.sum(_seg <= _dti_arr[_t]) / len(_seg)
_master["dti_pct_roll"] = _pct

# Add correction label (y=1 if >10% drawdown within 20q; last 20 rows become NaN).
_labeled = add_correction_label(_master)

# 1-quarter log return (used for strategy simulation and equity curves).
_labeled["ret_1q"] = np.log(_labeled["real_price_index"]).diff()

# Split.
train_df = _labeled[_labeled["date"] <= TRAIN_END].copy()
test_df  = _labeled[_labeled["date"] >= TEST_START].copy()

# ── FIREWALL ASSERTION: test set must not contain pre-2008 data ──
assert (test_df["date"] <= TRAIN_END).sum() == 0, \
    f"VIOLATION: test set contains {(test_df['date'] <= TRAIN_END).sum()} obs with date <= 2007Q4"
assert test_df["date"].min() >= TEST_START, \
    f"VIOLATION: earliest test obs is {test_df['date'].min().date()}, expected >= {TEST_START.date()}"

print("Date firewall: PASSED")
print()
print(f"Train : {len(train_df):3d} rows  "
      f"{train_df['date'].min().date()} — {train_df['date'].max().date()}")
print(f"Test  : {len(test_df):3d} rows  "
      f"{test_df['date'].min().date()} — {test_df['date'].max().date()}")
print(f"Cols  : {test_df.columns.tolist()}")
print()
print("Test window regime breakdown:")
_reg_counts = test_df["regime"].value_counts().sort_index()
print(f"  regime=0 (adverse)        : {_reg_counts.get(0, 0)} quarters")
print(f"  regime=1 (accommodative)  : {_reg_counts.get(1, 0)} quarters")
print()
print("Note: y labels absent for last 20 rows (add_correction_label horizon).")
print(f"      Labeled test rows: {test_df['y'].notna().sum()} / {len(test_df)}")

---
## Table 1: Cell counts (regime × supply condition)

**Supply condition:** `fragile` = DTI > `DTI_FRAGILITY_CUTOFF`; `normal` = DTI ≤ cutoff.  
Shown separately for train and test windows.  
Cells with n < 10 are flagged; conclusions from flagged cells should be interpreted with caution.

In [ ]:
# ── TABLE 1: Cell counts ──────────────────────────────────────

def _supply_label(dti_series):
    return np.where(dti_series > DTI_FRAGILITY_CUTOFF,
                    "fragile (DTI>cutoff)", "normal (DTI≤cutoff)")

def make_cell_counts(df, period_label):
    df = df.copy()
    df["supply"] = _supply_label(df["dti"])
    rows = []
    for reg, reg_name in [(0, "regime=0 (adverse)"), (1, "regime=1 (accommodative)")]:
        for sup in ["fragile (DTI>cutoff)", "normal (DTI≤cutoff)"]:
            mask = (df["regime"] == reg) & (df["supply"] == sup)
            n_obs   = int(mask.sum())
            # y may be NaN for the last 20 rows; count only labeled obs
            n_crash = int(df.loc[mask, "y"].sum(min_count=1)) if n_obs > 0 else 0
            n_labeled = int(df.loc[mask, "y"].notna().sum())
            rows.append({
                "period":   period_label,
                "regime":   reg_name,
                "supply":   sup,
                "n_obs":    n_obs,
                "n_labeled":n_labeled,
                "n_crash":  n_crash,
                "flag":     "⚠ n<10" if n_labeled < 10 else "",
            })
    return pd.DataFrame(rows)

table1 = pd.concat(
    [make_cell_counts(train_df, "Train (≤2007Q4)"),
     make_cell_counts(test_df,  "Test  (≥2008Q1)")],
    ignore_index=True
)

print("TABLE 1: Cell counts (regime × supply condition)")
print("-" * 65)
display(
    table1.set_index(["period", "regime", "supply"])
          .style.set_properties(**{"text-align": "right"})
)

---
## Table 2: Conditional crash frequency + Wilson 95% CI

Primary criterion 3 (PHASE_OOS_DESIGN §7.1): crash frequency in `regime=0, fragile` cell must
exceed `regime=0, normal` cell with **non-overlapping Wilson CIs** to support the hypothesis.  

Wilson CI is the primary uncertainty estimate (see §5.2). Do not summarise as significant/not significant — report the intervals and describe overlap.

In [ ]:
# ── TABLE 2: Crash frequency + Wilson 95% CI ─────────────────

def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom  = 1.0 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return (max(0.0, center - margin), min(1.0, center + margin))

def make_freq_table(df, period_label):
    df = df.copy()
    df["supply"] = _supply_label(df["dti"])
    # Use only labeled rows for crash frequency
    df = df.dropna(subset=["y"])
    rows = []
    for reg, reg_name in [(0, "regime=0 (adverse)"), (1, "regime=1 (accommodative)")]:
        for sup in ["fragile (DTI>cutoff)", "normal (DTI≤cutoff)"]:
            mask = (df["regime"] == reg) & (df["supply"] == sup)
            n = int(mask.sum())
            k = int(df.loc[mask, "y"].sum()) if n > 0 else 0
            freq = k / n if n > 0 else np.nan
            lo, hi = wilson_ci(k, n)
            rows.append({
                "period":      period_label,
                "regime":      reg_name,
                "supply":      sup,
                "n":           n,
                "k (crashes)": k,
                "freq":        round(freq, 4) if not np.isnan(freq) else np.nan,
                "CI lower":    round(lo, 4) if not np.isnan(lo) else np.nan,
                "CI upper":    round(hi, 4) if not np.isnan(hi) else np.nan,
                "flag":        "⚠ n<10" if n < 10 else "",
            })
    return pd.DataFrame(rows)

table2 = pd.concat(
    [make_freq_table(train_df, "Train (≤2007Q4)"),
     make_freq_table(test_df,  "Test  (≥2008Q1)")],
    ignore_index=True
)

print("TABLE 2: Conditional crash frequency + Wilson 95% CI")
print("-" * 70)
display(table2.set_index(["period", "regime", "supply"]))
print()
print("Primary criterion 3 check (test window):")
_test2 = table2[table2["period"].str.startswith("Test")].set_index(["regime", "supply"])
try:
    _fragile_lo = _test2.loc[("regime=0 (adverse)", "fragile (DTI>cutoff)"), "CI lower"]
    _fragile_hi = _test2.loc[("regime=0 (adverse)", "fragile (DTI>cutoff)"), "CI upper"]
    _normal_lo  = _test2.loc[("regime=0 (adverse)",  "normal (DTI≤cutoff)"),  "CI lower"]
    _normal_hi  = _test2.loc[("regime=0 (adverse)",  "normal (DTI≤cutoff)"),  "CI upper"]
    _overlap = _fragile_lo < _normal_hi and _normal_lo < _fragile_hi
    print(f"  regime=0 fragile  CI: [{_fragile_lo:.3f}, {_fragile_hi:.3f}]")
    print(f"  regime=0 normal   CI: [{_normal_lo:.3f}, {_normal_hi:.3f}]")
    print(f"  CIs overlap: {_overlap}  "
          f"({'inconclusive' if _overlap else 'non-overlapping — supports hypothesis'})")
except KeyError:
    print("  (insufficient data to evaluate primary criterion 3)")

---
## Table 3: Strategy comparison (test window)

Three strategies evaluated on the test window:

| Strategy | Rule |
|---|---|
| always_in | Fully invested every quarter |
| regime_overlay | Invested unless `regime==0 AND DTI > DTI_FRAGILITY_CUTOFF` |
| always_out | Never invested (return = 0 every quarter) |

Metrics per PHASE_OOS_DESIGN §5.3. Return and vol annualised (×4 and ×√4 respectively).

In [ ]:
# ── TABLE 3: Strategy comparison ─────────────────────────────
from src.backtests.evaluation import summarize, max_drawdown

# Use all test rows with valid 1-quarter returns.
_df3 = test_df.dropna(subset=["ret_1q"]).copy()
_df3["fragile_regime"] = (_df3["regime"] == 0) & (_df3["dti"] > DTI_FRAGILITY_CUTOFF)

_strategies = {
    "always_in":      _df3["ret_1q"],
    "regime_overlay": _df3["ret_1q"].where(~_df3["fragile_regime"], other=0.0),
    "always_out":     pd.Series(0.0, index=_df3.index),
}

table3_rows = []
for strat_name, ret in _strategies.items():
    s = summarize(ret, name=strat_name)
    time_in = float((ret != 0.0).mean())
    ann_ret = s["mean"] * 4
    ann_vol = s["vol"] * (4 ** 0.5)
    ann_sharpe = ann_ret / ann_vol if ann_vol > 1e-9 else np.nan
    table3_rows.append({
        "strategy":      strat_name,
        "ann. return":   round(ann_ret,    4),
        "ann. vol":      round(ann_vol,    4),
        "sharpe (ann)":  round(ann_sharpe, 4),
        "p05 (qtr)":     round(s["p05"],   4),
        "max drawdown":  round(s["maxdd"],  4),
        "time-in-mkt":   f"{time_in:.1%}",
    })

table3 = pd.DataFrame(table3_rows).set_index("strategy")

print("TABLE 3: Strategy comparison — test window (2008Q1 onwards)")
print("-" * 72)
display(table3)
print()
print("Primary criteria check:")
_ai  = table3.loc["always_in"]
_ov  = table3.loc["regime_overlay"]
_p1 = _ov["p05 (qtr)"] > _ai["p05 (qtr)"]
_p2 = _ov["max drawdown"] > _ai["max drawdown"]  # maxdd is negative; > means smaller magnitude
print(f"  Criterion 1 — overlay p05 > always_in p05:     {_p1}  "
      f"({_ov['p05 (qtr)']:.4f} vs {_ai['p05 (qtr)']:.4f})")
print(f"  Criterion 2 — overlay maxdd > always_in maxdd: {_p2}  "
      f"({_ov['max drawdown']:.4f} vs {_ai['max drawdown']:.4f})")

---
## Figure 1: Equity curves (test window)

Three strategies on one chart. Red shading marks adverse-regime quarters (`regime=0`).  
Navy shading marks the 2008–2009 crisis window.

In [ ]:
# ── FIGURE 1: Equity curves ───────────────────────────────────
Path("../../outputs/oos").mkdir(parents=True, exist_ok=True)

_df_plot = _df3.reset_index(drop=True)
_colors  = {"always_in": "#2166ac", "regime_overlay": "#d6604d", "always_out": "#888888"}
_labels  = {"always_in": "Always-in", "regime_overlay": "Regime overlay", "always_out": "Always-out"}

fig, (ax_eq, ax_reg) = plt.subplots(
    2, 1, figsize=(13, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 0.7]}
)

# ── Equity curves ────────────────────────────────────────────
for name, ret in _strategies.items():
    _ret = ret.reindex(_df_plot.index).fillna(0.0)
    _cum = np.exp(np.cumsum(_ret.values))
    _cum = _cum / _cum[0]  # re-anchor to 1.0
    ax_eq.plot(_df_plot["date"], _cum, label=_labels[name],
               color=_colors[name], lw=2.0)

# ── Regime shading (adverse = red band) ──────────────────────
_dates = _df_plot["date"].values
_reg   = _df_plot["regime"].values
_in_adverse = False
_band_start = None
for i, (dt, r) in enumerate(zip(_dates, _reg)):
    if r == 0 and not _in_adverse:
        _band_start  = dt
        _in_adverse  = True
    elif r != 0 and _in_adverse:
        ax_eq.axvspan(_band_start, dt, alpha=0.10, color="tomato", zorder=0)
        _in_adverse = False
if _in_adverse:
    ax_eq.axvspan(_band_start, _dates[-1], alpha=0.10, color="tomato", zorder=0)

# ── GFC annotation ───────────────────────────────────────────
_gfc_start = pd.Timestamp("2008-01-01")
_gfc_end   = pd.Timestamp("2009-06-30")
ax_eq.axvspan(_gfc_start, _gfc_end, alpha=0.06, color="navy", zorder=0)
ax_eq.annotate(
    "GFC\n2008–09",
    xy=(pd.Timestamp("2008-10-01"), ax_eq.get_ylim()[0] + 0.02),
    fontsize=8, color="navy", ha="center", va="bottom",
)

# ── 2022 rate-rise annotation ─────────────────────────────────
ax_eq.axvline(pd.Timestamp("2022-03-01"), color="darkorange", ls=":", lw=1.2,
              label="2022 rate rise")

ax_eq.axhline(1.0, color="black", lw=0.5, ls="--", alpha=0.5)
ax_eq.set_ylabel("Cumulative return (base = 1.0)", fontsize=10)
ax_eq.set_title(
    "Figure 1: Strategy Equity Curves — Test Window (2008Q1 onwards)\n"
    "Red shading = adverse regime (real rate ≥ 0)  ·  Navy = GFC window",
    fontsize=11
)
ax_eq.legend(fontsize=9, loc="upper left")
ax_eq.grid(axis="y", alpha=0.3)

# ── Regime indicator strip ────────────────────────────────────
ax_reg.fill_between(
    _df_plot["date"], 0, (1 - _df_plot["regime"]).values,
    step="mid", color="tomato", alpha=0.6, label="adverse (rate ≥ 0)"
)
ax_reg.fill_between(
    _df_plot["date"], 0, _df_plot["regime"].values,
    step="mid", color="steelblue", alpha=0.4, label="accommodative"
)
ax_reg.set_yticks([])
ax_reg.set_ylabel("Regime", fontsize=9)
ax_reg.set_xlabel("Date", fontsize=10)
ax_reg.legend(fontsize=8, loc="upper right", ncol=2)

plt.tight_layout()
_fig1_path = "../../outputs/oos/equity_curve.png"
plt.savefig(_fig1_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig1_path}")

---
## Figure 2: Threshold sensitivity (DTI cutoff ±20%)

Re-evaluates the regime overlay strategy with DTI cutoffs at 0.8×, 0.9×, 1.0×, 1.1×, 1.2× of
`DTI_FRAGILITY_CUTOFF`. Purpose: confirm results are not knife-edge sensitive to the exact cutoff.  

**Do not select a new cutoff based on these results.** Per PHASE_OOS_DESIGN §5.5, this is a
robustness check only.

In [ ]:
# ── FIGURE 2: Threshold sensitivity ──────────────────────────
_multipliers = [0.8, 0.9, 1.0, 1.1, 1.2]
_sens_rows = []

for _mult in _multipliers:
    _cutoff = DTI_FRAGILITY_CUTOFF * _mult
    _df_s = _df3.copy()
    _df_s["fragile"] = (_df_s["regime"] == 0) & (_df_s["dti"] > _cutoff)
    _ret_s = _df_s["ret_1q"].where(~_df_s["fragile"], other=0.0)
    _s = summarize(_ret_s)
    _ann_ret = _s["mean"] * 4
    _ann_vol = _s["vol"] * (4 ** 0.5)
    _ann_sharpe = _ann_ret / _ann_vol if _ann_vol > 1e-9 else np.nan
    _time_in = float((_ret_s != 0.0).mean())
    _sens_rows.append({
        "mult":           f"{_mult:.1f}×",
        "DTI cutoff":     round(_cutoff, 2),
        "ann. return":    round(_ann_ret,    4),
        "ann. vol":       round(_ann_vol,    4),
        "sharpe (ann)":   round(_ann_sharpe, 4),
        "p05 (qtr)":      round(_s["p05"],   4),
        "max drawdown":   round(_s["maxdd"],  4),
        "time-in-mkt":    round(_time_in,     3),
    })

sens_df = pd.DataFrame(_sens_rows).set_index("mult")

print("FIGURE 2 (table): Threshold sensitivity — regime overlay at ±20% DTI cutoff")
print("-" * 72)
display(sens_df)

# ── Sensitivity chart ─────────────────────────────────────────
fig2, axes = plt.subplots(2, 2, figsize=(12, 7), sharey=False)
_x = _multipliers
_met = [
    ("ann. return",   "Annualised Return",        "#2166ac"),
    ("p05 (qtr)",     "p05 (quarterly)",           "#d6604d"),
    ("max drawdown",  "Max Drawdown",              "#762a83"),
    ("time-in-mkt",   "Time in Market (fraction)", "#4dac26"),
]
for ax, (col, title, color) in zip(axes.flat, _met):
    _vals = sens_df[col].values.astype(float)
    ax.plot(_x, _vals, marker="o", color=color, lw=2.0, ms=6)
    ax.axvline(1.0, color="black", ls="--", lw=1, alpha=0.5, label="base (1.0×)")
    # Highlight base value
    _base_val = sens_df.loc["1.0×", col]
    ax.scatter([1.0], [_base_val], color="black", zorder=5, s=50)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Cutoff multiplier", fontsize=9)
    ax.set_xticks(_x)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

fig2.suptitle(
    "Figure 2: Sensitivity of Regime Overlay to DTI Fragility Cutoff (±20%)\n"
    "Black dashed = frozen cutoff (1.0×). Stable lines indicate robust results.",
    fontsize=11
)
plt.tight_layout()
_fig2_path = "../../outputs/oos/threshold_sensitivity.png"
plt.savefig(_fig2_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig2_path}")
print()
print("Sensitivity robustness check:")
_p05_range    = sens_df["p05 (qtr)"].max() - sens_df["p05 (qtr)"].min()
_maxdd_range  = sens_df["max drawdown"].max() - sens_df["max drawdown"].min()
_sharpe_range = sens_df["sharpe (ann)"].max() - sens_df["sharpe (ann)"].min()
print(f"  p05 range across ±20% cutoffs:    {_p05_range:.4f}")
print(f"  maxdd range across ±20% cutoffs:  {_maxdd_range:.4f}")
print(f"  sharpe range across ±20% cutoffs: {_sharpe_range:.4f}")
print("  (small ranges = results robust to exact cutoff choice)")